In [6]:
# # --- Standard visualization for reduced PAMAP2 features ---
# import matplotlib.pyplot as plt
# import seaborn as sns
# from collections import Counter

# # 1) Print initial vs final features
# print("Initial raw channels (54):", colnames()[:15], "... total:", len(colnames()))
# print("Reduced base channels used (after config):", base_cols)
# print("Stats used:", STATS)
# print("Final features per window:", len(feature_names))

# # 2) Class balance
# class_counts = Counter(y)
# print("Class balance (0=walking, 1=rope):", class_counts)

# plt.figure()
# sns.countplot(x=y)
# plt.title("Class Distribution: Walking vs Rope Jumping")
# plt.xticks([0,1], ["Walking","Rope Jumping"])
# plt.ylabel("Window count")
# plt.show()

# # 3) PCA scatter
# Xp2 = pca.transform(Xz)[:, :3]  # first 3 comps
# plt.figure(figsize=(6,5))
# sns.scatterplot(x=Xp2[:,0], y=Xp2[:,1], hue=y, alpha=0.6, palette={0:"blue",1:"red"})
# plt.title("PCA Scatter Plot (PC1 vs PC2)")
# plt.xlabel("PC1"); plt.ylabel("PC2")
# plt.legend(title="Class", labels=["Walking","Rope"])
# plt.show()

# # Optional 3D scatter
# from mpl_toolkits.mplot3d import Axes3D
# fig = plt.figure(figsize=(7,6))
# ax = fig.add_subplot(111, projection="3d")
# ax.scatter(Xp2[:,0], Xp2[:,1], Xp2[:,2], c=y, cmap="bwr", alpha=0.5)
# ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
# ax.set_title("3D PCA Scatter")
# plt.show()

# # 4) Heart rate distribution
# if "heart_rate" in base_cols:
#     feat_df["y"] = y
#     plt.figure()
#     sns.boxplot(x="y", y="heart_rate__std", data=feat_df.replace(0, np.nan))  # example stat
#     plt.xticks([0,1], ["Walking","Rope"])
#     plt.title("Heart Rate Variability (Std) by Class")
#     plt.show()

# # 5) Feature distributions
# # Pick a few interesting features (std/rms of acc/gyro)
# chosen_feats = [c for c in feature_names if "__std" in c or "__rms" in c][:6]

# fig, axes = plt.subplots(2, 3, figsize=(12,6))
# for ax, feat in zip(axes.ravel(), chosen_feats):
#     sns.kdeplot(x=feat_df[feat], hue=y, common_norm=False, palette={0:"blue",1:"red"}, ax=ax)
#     ax.set_title(feat)
# plt.suptitle("Feature Distributions (Walking vs Rope)", y=1.02)
# plt.tight_layout()
# plt.show()


In [7]:
# # --- Baseline Logistic Regression with LOSO CV on reduced features ---

# import re
# from pathlib import Path
# import numpy as np
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
# from sklearn.model_selection import GroupKFold
# import matplotlib.pyplot as plt

# # ---- CONFIG ----
# DATA_DIR = Path("/home/epigou/cs_9170_project/PAMAP2_Dataset")
# FS = 100
# WIN_S, STEP_S = 5.0, 2.5
# WIN, STEP = int(WIN_S*FS), int(STEP_S*FS)

# DROP_MAG = True
# USE_VECTOR_NORMS = True
# STATS = ("std","rms")

# # ---- helpers ----
# def colnames():
#     cols = ["timestamp","activity_id","heart_rate"]
#     imu_pos = ["hand","chest","ankle"]
#     sub = ["temp","acc16g_x","acc16g_y","acc16g_z","acc6g_x","acc6g_y","acc6g_z",
#            "gyro_x","gyro_y","gyro_z","mag_x","mag_y","mag_z",
#            "orient_w","orient_x","orient_y","orient_z"]
#     for p in imu_pos:
#         cols += [f"{p}_{s}" for s in sub]
#     return cols

# def parse_sid(p):
#     m = re.search(r"subject(\d+)", p.stem.lower())
#     return m.group(1) if m else "unknown"

# def load_all():
#     files=[]
#     for sub in ["Protocol","Optional","protocol","optional"]:
#         d = DATA_DIR/sub
#         if d.exists(): files += list(d.glob("subject*.dat"))
#     dfs=[]
#     for f in sorted(files):
#         df = pd.read_csv(f, sep=r"\s+", header=None, names=colnames(),
#                          engine="python", na_values=["NaN","nan"])
#         df["subject_id"]=parse_sid(f)
#         dfs.append(df)
#     return pd.concat(dfs, ignore_index=True)

# def to_binary(df):
#     df = df[df["activity_id"].isin([4,24])].copy()
#     df["y"] = df["activity_id"].map({4:0, 24:1})
#     return df

# def interpolate_by_subject(df):
#     num = df.select_dtypes(include=[np.number]).columns.tolist()
#     num = [c for c in num if c not in ("activity_id","y")]
#     return df.sort_values(["subject_id","timestamp"]).groupby("subject_id", group_keys=False).apply(
#         lambda g: g.assign(**{c: g[c].interpolate(limit_direction="both") for c in num})
#     )

# # ---- feature reduction ----
# IMU_POS = ["hand","chest","ankle"]
# ACC = ["acc16g_x","acc16g_y","acc16g_z"]
# GYR = ["gyro_x","gyro_y","gyro_z"]
# MAG = ["mag_x","mag_y","mag_z"]

# def build_channel_plan():
#     plan = {"heart_rate": [("scalar","heart_rate")]}
#     sensors = [("acc", ACC), ("gyr", GYR)]
#     if not DROP_MAG:
#         sensors.append(("mag", MAG))
#     if USE_VECTOR_NORMS:
#         for pos in IMU_POS:
#             for name, axes in sensors:
#                 cols = [f"{pos}_{a}" for a in axes]
#                 plan[f"{pos}_{name}_norm"] = [("norm", cols)]
#     else:
#         for pos in IMU_POS:
#             for name, axes in sensors:
#                 for a in axes:
#                     plan[f"{pos}_{a}"] = [("scalar", f"{pos}_{a}")]
#     return plan

# PLAN = build_channel_plan()

# def compute_base_signals(df):
#     out = pd.DataFrame(index=df.index)
#     for key, ops in PLAN.items():
#         kind, cols = ops[0]
#         if kind=="scalar":
#             out[key]=df[cols]
#         else:
#             M=df[cols].values
#             out[key]=np.sqrt((M**2).sum(axis=1))
#     return out

# def window_features(df):
#     df=df.sort_values(["subject_id","timestamp"]).reset_index(drop=True)
#     base=compute_base_signals(df)
#     base["subject_id"]=df["subject_id"].values
#     base["y"]=df["y"].values
#     rows, labels, sids=[],[],[]
#     base_cols=[c for c in base.columns if c not in("subject_id","y")]
#     for sid,g in base.groupby("subject_id", sort=False):
#         g=g.reset_index(drop=True)
#         for start in range(0, max(0,len(g)-WIN+1), int(WIN/2)):
#             w=g.iloc[start:start+WIN]
#             if len(w)<WIN: continue
#             y=int(np.round(w["y"].values.mean()))
#             W=w[base_cols]
#             feats={}
#             if "mean" in STATS:
#                 feats.update(W.mean().add_suffix("__mean").to_dict())
#             if "std" in STATS:
#                 feats.update(W.std(ddof=1).add_suffix("__std").to_dict())
#             if "rms" in STATS:
#                 feats.update((np.sqrt((W**2).mean())).add_suffix("__rms").to_dict())
#             feats["subject_id"]=sid
#             rows.append(feats); labels.append(y); sids.append(sid)
#     feat_df=pd.DataFrame(rows).fillna(0.0)
#     return feat_df, np.array(labels), np.array(sids)

# # ---- pipeline ----
# df=load_all()
# df=to_binary(df)
# df=interpolate_by_subject(df)
# feat_df,y,subjects=window_features(df)

# X=feat_df.drop(columns=["subject_id"]).values
# print("Windows:",len(y)," Features per window:",X.shape[1])

# # ---- LOSO CV ----
# cm_total=np.zeros((2,2),dtype=int)
# metrics=[]
# cv=GroupKFold(n_splits=len(np.unique(subjects)))
# for tr,te in cv.split(X,y,groups=subjects):
#     Xtr,Xte=X[tr],X[te]; ytr,yte=y[tr],y[te]
#     scaler=StandardScaler().fit(Xtr)
#     Xtr_z=scaler.transform(Xtr); Xte_z=scaler.transform(Xte)
#     pca=PCA(n_components=0.95,svd_solver="full").fit(Xtr_z)
#     Xtr_p=pca.transform(Xtr_z); Xte_p=pca.transform(Xte_z)
#     clf=LogisticRegression(max_iter=2000).fit(Xtr_p,ytr)
#     yhat=clf.predict(Xte_p)
#     cm=confusion_matrix(yte,yhat,labels=[0,1])
#     cm_total+=cm
#     prec,rec,f1,_=precision_recall_fscore_support(yte,yhat,labels=[0,1],zero_division=0)
#     acc=accuracy_score(yte,yhat)
#     metrics.append([acc,prec[1],rec[1],f1[1]])

# metrics=np.array(metrics)
# print("Mean metrics across subjects:")
# print("Accuracy:",metrics[:,0].mean().round(3),
#       " Precision(min):",metrics[:,1].mean().round(3),
#       " Recall(min):",metrics[:,2].mean().round(3),
#       " F1(min):",metrics[:,3].mean().round(3))

# # ---- Confusion Matrix ----
# cmn=cm_total.astype(float)/cm_total.sum(axis=1,keepdims=True)
# fig,ax=plt.subplots()
# im=ax.imshow(cmn,interpolation='nearest',cmap="Blues")
# ax.figure.colorbar(im,ax=ax)
# ax.set(xticks=[0,1],yticks=[0,1],
#        xticklabels=["Walking","Rope"],yticklabels=["Walking","Rope"],
#        ylabel="True label",xlabel="Predicted label",
#        title="Aggregated Confusion Matrix (LOSO)")
# for i in range(2):
#     for j in range(2):
#         ax.text(j,i,f"{cm_total[i,j]}\n({cmn[i,j]:.2f})",
#                 ha="center",va="center",
#                 color="white" if cmn[i,j]>0.5 else "black")
# plt.tight_layout()
# plt.show()


In [ ]:
def split_pamap2_binary(
    data_dir="/home/epigou/cs_9170_project/PAMAP2_Dataset",
    train_size=None,
    bias_pct=0.20,
    val_frac=0.20,
    test_frac=0.20,
    seed=None,
    pca_variance=0.95,
    win_seconds=5.0,
    step_seconds=2.5,
    drop_magnetometers=True,
    use_vector_norms=True,
    stats=("std", "rms"),
):
    import re
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    import torch

    assert 0 < val_frac < 1 and 0 < test_frac < 1 and (val_frac + test_frac) < 1, \
        "val_frac and test_frac must be in (0,1) and sum to < 1."

    rng = np.random.RandomState(seed)

    # -------- constants / helpers --------
    FS = 100
    WIN = int(win_seconds * FS)
    STEP = int(step_seconds * FS)

    imu_pos = ["hand", "chest", "ankle"]
    ACC16 = ["acc16g_x", "acc16g_y", "acc16g_z"]
    GYR = ["gyro_x", "gyro_y", "gyro_z"]
    MAG = ["mag_x", "mag_y", "mag_z"]

    def colnames():
        cols = ["timestamp", "activity_id", "heart_rate"]
        sub = ["temp","acc16g_x","acc16g_y","acc16g_z","acc6g_x","acc6g_y","acc6g_z",
               "gyro_x","gyro_y","gyro_z","mag_x","mag_y","mag_z","orient_w","orient_x","orient_y","orient_z"]
        for p in imu_pos:
            cols += [f"{p}_{s}" for s in sub]
        return cols  # 54 total

    def parse_sid(path):
        m = re.search(r"subject(\d+)", Path(path).stem.lower())
        return m.group(1) if m else "unknown"

    # -------- load all .dat --------
    data_dir = Path(data_dir)
    files = []
    for sub in ["Protocol", "Optional", "protocol", "optional"]:
        d = data_dir / sub
        if d.exists():
            files += list(d.glob("subject*.dat"))
    if not files:
        raise FileNotFoundError(f"No .dat files found under {data_dir}")

    dfs = []
    for f in sorted(files):
        df = pd.read_csv(
            f, sep=r"\s+", header=None, names=colnames(),
            engine="python", na_values=["NaN","nan"]
        )
        df["subject_id"] = parse_sid(f)
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)

    # -------- filter to two activities & map labels --------
    df = df[df["activity_id"].isin([4, 24])].copy()
    df["y"] = df["activity_id"].map({4: 0, 24: 1})

    # -------- per-subject interpolation (avoid leakage) --------
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    # do not interpolate labels
    num = [c for c in num if c not in ("activity_id", "y")]
    df = df.sort_values(["subject_id", "timestamp"]).groupby("subject_id", group_keys=False).apply(
        lambda g: g.assign(**{c: g[c].interpolate(limit_direction="both") for c in num})
    )

    # -------- base channels (heart + acc16g + gyro +/- mag) --------
    keep_triplets = [("acc", ACC16), ("gyr", GYR)]
    if not drop_magnetometers:
        keep_triplets.append(("mag", MAG))

    base_cols = []
    if use_vector_norms:
        # vector norms per IMU & sensor
        for p in imu_pos:
            for name, axes in keep_triplets:
                cols = [f"{p}_{a}" for a in axes]
                df[f"{p}_{name}_norm"] = np.sqrt((df[cols].values ** 2).sum(axis=1))
                base_cols.append(f"{p}_{name}_norm")
    else:
        # keep raw axes (x, y, z)
        for p in imu_pos:
            for _, axes in keep_triplets:
                for a in axes:
                    base_cols.append(f"{p}_{a}")

    base_cols = ["heart_rate"] + base_cols  # prepend HR

    # -------- windowing → features (std, rms) --------
    rows, labels, subjects = [], [], []
    df = df.sort_values(["subject_id", "timestamp"]).reset_index(drop=True)

    # which stats are we computing?
    compute_mean = ("mean" in stats)
    compute_std  = ("std"  in stats)
    compute_rms  = ("rms"  in stats)

    for sid, g in df.groupby("subject_id", sort=False):
        g = g.reset_index(drop=True)
        n = len(g)
        for start in range(0, max(0, n - WIN + 1), STEP):
            w = g.iloc[start:start + WIN]
            if len(w) < WIN:
                continue
            # binary majority label in window
            y_win = int(np.round(w["y"].values.mean()))
            W = w[base_cols]

            feats = {}
            if compute_mean:
                feats.update(W.mean().add_suffix("__mean").to_dict())
            if compute_std:
                feats.update(W.std(ddof=1).add_suffix("__std").to_dict())
            if compute_rms:
                feats.update((np.sqrt((W**2).mean())).add_suffix("__rms").to_dict())

            feats["subject_id"] = sid
            rows.append(feats); labels.append(y_win); subjects.append(sid)

    feat_df = pd.DataFrame(rows).fillna(0.0)
    y_all = np.asarray(labels, dtype=int)
    subj_all = np.asarray(subjects)

    # -------- subject-aware split (train / val / test) --------
    unique_subjects = np.unique(subj_all)
    rng.shuffle(unique_subjects)

    n_subj = len(unique_subjects)
    n_test = max(1, int(round(test_frac * n_subj)))
    n_val  = max(1, int(round(val_frac  * n_subj)))
    # keep the rest for train
    if n_test + n_val >= n_subj:
        # ensure at least 1 train subject
        n_test = max(1, n_test)
        n_val  = max(1, min(n_val, n_subj - n_test - 1))

    test_subj = unique_subjects[:n_test]
    val_subj  = unique_subjects[n_test:n_test + n_val]
    train_subj= unique_subjects[n_test + n_val:]

    def mask_for(subj_list):
        return np.isin(subj_all, subj_list)

    m_train, m_val, m_test = mask_for(train_subj), mask_for(val_subj), mask_for(test_subj)

    X_train_df = feat_df[m_train].copy()
    X_val_df   = feat_df[m_val].copy()
    X_test_df  = feat_df[m_test].copy()
    y_train    = y_all[m_train]
    y_val      = y_all[m_val]
    y_test     = y_all[m_test]

    # -------- apply the SAME bias inside each split --------
    def apply_bias(df_split, y_split, target_minority_pct, seed_local):
        dfb = df_split.copy()
        dfb["__y__"] = y_split
        dfM = dfb[dfb["__y__"] == 0]
        dfm = dfb[dfb["__y__"] == 1]

        nM = len(dfM); nm = len(dfm)
        if nM == 0 or nm == 0:
            out = dfb
        else:
            # target_minority_pct = nm_kept / (nM + nm_kept)
            # => nm_kept = (target * nM) / (1 - target)
            nm_keep = int(np.floor((target_minority_pct * nM) / (1 - target_minority_pct)))
            nm_keep = min(nm, max(1, nm_keep))
            dfm_b = dfm.sample(n=nm_keep, random_state=seed_local, replace=False)
            out = pd.concat([dfM, dfm_b], axis=0).sample(frac=1.0, random_state=seed_local).reset_index(drop=True)

        y_out = out["__y__"].to_numpy(dtype=int)
        X_out = out.drop(columns=["__y__"])
        return X_out, y_out

    target_minority_pct = float(bias_pct)
    X_train_biased_df, y_train_biased = apply_bias(X_train_df, y_train, target_minority_pct, rng.randint(0, 10**6))
    X_val_biased_df,   y_val_biased   = apply_bias(X_val_df,   y_val,   target_minority_pct, rng.randint(0, 10**6))
    X_test_biased_df,  y_test_biased  = apply_bias(X_test_df,  y_test,  target_minority_pct, rng.randint(0, 10**6))

    # Optional: subsample TRAIN after biasing
    if train_size is not None and train_size < len(X_train_biased_df):
        X_train_biased_df, _, y_train_biased, _ = train_test_split(
            X_train_biased_df, y_train_biased,
            train_size=train_size, random_state=rng.randint(0, 10**6), stratify=y_train_biased
        )

    # -------- scaler + PCA on TRAIN only --------
    meta_cols = ["subject_id"]
    feature_cols = [c for c in X_train_biased_df.columns if c not in meta_cols]

    scaler = StandardScaler()
    Xtr_z = scaler.fit_transform(X_train_biased_df[feature_cols].values)
    pca = PCA(n_components=pca_variance, svd_solver="full", random_state=rng.randint(0, 10**6))
    Xtr_p = pca.fit_transform(Xtr_z)

    def transform(df_split):
        Xz = scaler.transform(df_split[feature_cols].values)
        return pca.transform(Xz)

    Xval_p = transform(X_val_biased_df)
    Xtest_p= transform(X_test_biased_df)

    # -------- tensors on device --------
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    X_train_theta = torch.tensor(Xtr_p,   dtype=torch.float32, device=device)
    X_val_theta   = torch.tensor(Xval_p,  dtype=torch.float32, device=device)
    X_test_theta  = torch.tensor(Xtest_p, dtype=torch.float32, device=device)

    y_train_theta = torch.tensor(y_train_biased, dtype=torch.long, device=device)
    y_val_theta   = torch.tensor(y_val_biased,   dtype=torch.long, device=device)
    y_test_theta  = torch.tensor(y_test_biased,  dtype=torch.long, device=device)

    # ---- Sanity logging ----
    def log_dist(name, ysplit):
        n = len(ysplit); m = int((ysplit == 1).sum())
        pct = 100.0 * m / n if n else 0.0
        print(f"[{name}] size={n}, minority={m} ({pct:.2f}%)")
    log_dist("TRAIN", y_train_biased)
    log_dist("VAL",   y_val_biased)
    log_dist("TEST",  y_test_biased)
    print(f"PCA comps: {Xtr_p.shape[1]} (variance ≥ {pca_variance}) | Features in: {len(feature_cols)}")

    # Print the head of the final feature dataframe for inspection
    print("Feature DataFrame (first 5 rows):")
    print(feat_df.head())

    return X_train_theta, X_val_theta, X_test_theta, y_train_theta, y_val_theta, y_test_theta


In [11]:
split_pamap2_binary()

/tmp/ipykernel_85934/305673034.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.sort_values(["subject_id", "timestamp"]).groupby("subject_id", group_keys=False).apply(


[TRAIN] size=588, minority=117 (19.90%)
[VAL] size=217, minority=0 (0.00%)
[TEST] size=291, minority=30 (10.31%)
PCA comps: 6 (variance ≥ 0.95) | Features in: 14
Feature DataFrame (first 5 rows):
   heart_rate__std  hand_acc_norm__std  hand_gyr_norm__std  \
0         0.000000            0.960714            0.602770   
1         0.356232            2.930598            1.427951   
2         0.472328            4.604665            1.706445   
3         0.000000            4.731691            1.544008   
4         0.224838            4.219345            1.396609   

   chest_acc_norm__std  chest_gyr_norm__std  ankle_acc_norm__std  \
0             0.147995             0.217620             0.084870   
1             1.550353             0.210125             5.276006   
2             2.985106             0.225136             8.871426   
3             3.622124             0.246760             9.995264   
4             3.678687             0.241996            10.607191   

   ankle_gyr_norm__std

(tensor([[-0.9119,  0.8720,  0.6955, -0.5767,  0.0654,  0.1472],
         [-0.6355,  0.9322,  1.0346,  0.2902,  0.0942, -0.1624],
         [ 0.0668,  0.8321, -0.0428,  0.7336, -0.4211, -0.7467],
         ...,
         [-0.9375,  0.8356,  0.8971, -0.5319,  0.0279,  0.0655],
         [-1.1579,  0.4403,  1.0516, -0.6200,  0.1288,  0.1271],
         [-1.2297, -0.0362, -0.6410,  1.4049, -0.0843, -0.4819]],
        device='cuda:0'),
 tensor([[-3.4327e+00, -1.8778e+00,  4.0285e-02, -1.1667e+00,  3.7342e-02,
           3.7978e-01],
         [-3.1636e+00, -1.3286e+00, -6.5364e-01, -4.9611e-01,  2.1371e-01,
           3.5565e-01],
         [-2.2824e+00, -7.9091e-01, -3.5771e-01, -3.7515e-01, -4.1663e-03,
           2.9951e-01],
         ...,
         [-2.3084e+00, -6.9388e-02, -2.6948e-01,  1.5790e-01, -7.3981e-01,
           4.1445e-01],
         [-1.7307e+00,  3.2813e-01, -2.2501e-01,  1.0445e-01, -9.8486e-01,
           2.4982e-01],
         [-1.6776e+00, -3.9749e+00, -2.0714e+00,  1.0917e+01